# Diabetic retinopathy staging with RETFound (auto-detects MAE vs DINOv2)

Detects which architecture your checkpoint actually is (RETFound-MAE, patch
size 16, vs RETFound-DINOv2, patch size 14) from its weight shapes, builds
that one using the authors' own `models_vit.py` (downloaded on first run so
there's no risk of a subtle reimplementation mismatch), and scores DR stage
(0-4, ICDR scale) on a fundus image. Also includes a ViT-adapted Grad-CAM
explainer.

All the logic lives in the `retfound_dr_pkg/` package next to this notebook -
this notebook is just a thin driver over it. See that package's modules for
the implementation details.

**One-time install:** `pip install torch torchvision timm pillow requests matplotlib numpy`

Re-run the setup cell once per kernel session, then use `scorer.score(...)` /
`scorer.explain(...)` as many times as you like - the model loads once and
stays cached.

In [ ]:
# ==== EDIT THIS if your checkpoint lives somewhere else ====
# Defaults to retfound_mae/checkpoint-best.pth (next to this notebook).
from retfound_dr_pkg import RetfoundDRScorer, DR_LABELS

scorer = RetfoundDRScorer()

## Check an image

Edit `image_path` below (use a raw string `r"..."` or forward slashes for
Windows paths), then re-run this cell for each image you want to check.
The model loads once on the first run and stays cached, so later checks
are fast.

In [ ]:
image_path = r"C:\Users\adity\Desktop\manit_assignmentes\mldl\mldl_lab\assignment_3\xai-dr-screening\dataset_classificiation\train_images\train_images\1b8ad0afe9fb.png"

result = scorer.score(image_path)
print("Predicted stage:", result["predicted_stage"])
print("\nFull probability breakdown:")
for label, p in result["probabilities"].items():
    print(f"  {label}: {p:.1%}")

## Grad-CAM (which part of the retina drove the prediction)

Standard Grad-CAM needs a convolutional spatial feature map, which a Vision
Transformer doesn't have. The ViT-adapted version used here instead (see
`retfound_dr_pkg/gradcam.py`):

1. Hooks the output of the **last transformer block** (`model.blocks[-1]`),
   shape `[1, num_tokens, D]` (cls/register tokens + one token per 16x16 patch).
2. Backprops the predicted (or a chosen) class logit to get the gradient of
   that activation.
3. Drops the non-patch (cls/register) tokens, average-pools the gradient
   over the remaining patch tokens to get a per-channel importance weight
   (this is the Grad-CAM weighting step, with "spatial location" = patch
   token instead of conv pixel), then does a weighted sum over channels per
   token - same math as classic Grad-CAM, just on a 14x14 patch grid instead
   of a conv grid.
4. ReLU's the result, reshapes the 14x14 grid, and bicubic-upsamples it back
   to 224x224 to overlay on the image.

This matches how `global_pool=True` actually classifies here too: the head
is fed the mean-pooled **patch** tokens (`forward_features` in `models_vit.py`
explicitly drops the cls token before pooling), so explaining via patch
tokens - not the cls token - is the right target.

Backprop needs more GPU memory than the forward-only inference in
`scorer.score()` (activations from all blocks have to be kept around for the
backward pass). If your GPU runs out of memory, this falls back to CPU
automatically - it'll just be slower.

In [ ]:
gradcam_result = scorer.explain(image_path)